# **Baseline Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
# <Student to fill this section>
group_name = "Group 17"
student_name = "Chen-Tai Wang"
student_id = "25976996"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [16]:
! pip install scikit-learn

### 0.b Import Packages

In [7]:
import pandas as pd
import altair as alt
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

---
## A. Assess Baseline Model

In [8]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

### A.1 Generate Predictions with Baseline Model

In [9]:
# Baseline: DummyClassifier — always predicts the most frequent class
baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train.values.ravel())

y_train_pred = baseline.predict(X_train)
y_val_pred = baseline.predict(X_val)
y_test_pred = baseline.predict(X_test)

print(f"Baseline strategy: always predict class {baseline.classes_[baseline.class_prior_.argmax()]}")
print(f"Class distribution (train): {y_train.value_counts().to_dict()}")

Baseline strategy: always predict class 0
Class distribution (train): {(0,): 8159, (1,): 5231}


### A.2 Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [10]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Calculate baseline metrics
for name, y_true, y_pred in [('Train', y_train, y_train_pred), ('Val', y_val, y_val_pred)]:
    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 (macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1 (class 1): {f1_score(y_true, y_pred, pos_label=1):.4f}")


=== Train ===
Accuracy: 0.6093
F1 (macro): 0.3786
F1 (class 1): 0.0000

=== Val ===
Accuracy: 0.6092
F1 (macro): 0.3786
F1 (class 1): 0.0000


In [11]:
performance_metrics_explanations = """
Selected metrics for this classification task:

1. Accuracy: Overall correct predictions. Useful as a baseline reference, but can be misleading with imbalanced classes (61/39 split).

2. F1 Score (macro): Harmonic mean of precision and recall, averaged equally across both classes. Chosen because both false positives (wasting marketing budget on customers who would repeat anyway)
and false negatives (missing at-risk customers) have business cost.

3. F1 Score (class 1 — repeat buyers): Specifically measures how well we identify repeat customers, which is the minority class of interest.

4. ROC-AUC (for trained models): Measures discrimination ability across all thresholds, providing a threshold-independent evaluation.

F1 macro is the primary metric as it balances performance across both classes.
"""

In [12]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance

> Provide some explanations on model performance


In [13]:
print("=== Baseline Performance (Validation Set) ===")
print(classification_report(y_val, y_val_pred, target_names=['Single Purchase', 'Repeat Buyer']))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

=== Baseline Performance (Validation Set) ===
                 precision    recall  f1-score   support

Single Purchase       0.61      1.00      0.76      1743
   Repeat Buyer       0.00      0.00      0.00      1118

       accuracy                           0.61      2861
      macro avg       0.30      0.50      0.38      2861
   weighted avg       0.37      0.61      0.46      2861


Confusion Matrix:
[[1743    0]
 [1118    0]]


In [14]:
baseline_performance_explanations = """
The baseline model (always predict majority class 'Single Purchase') achieves:
- Accuracy: ~60.9% (simply reflects the class distribution)
- F1 (macro): ~0.38 (very poor — cannot identify any repeat buyers)
- F1 (class 1): 0.00 (completely fails to predict repeat purchases)

This baseline establishes the minimum performance threshold. Any useful model must significantly exceed 60.9% accuracy AND achieve meaningful F1 for the repeat buyer class.
The baseline confirms this is not a trivially solvable problem and justifies building a machine learning model.
"""

In [15]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='baseline_performance_explanations', value=baseline_performance_explanations)